# Train model RVC giọng của bạn (Colab GPU)

Chạy lần lượt các cell. Yêu cầu: **Runtime → Change runtime type → T4 GPU**.

Chuẩn bị trước: chạy `scripts/prepare_dataset.py` ở máy local để cắt bản thu thành các đoạn 4-10 giây, rồi zip thư mục `dataset/` lại (khuyến nghị 10-30 phút audio sạch, không nhạc nền, không reverb, cùng một mic).

In [ ]:
!nvidia-smi
!python --version

## 1. Cài RVC WebUI

Nhánh main của RVC nhắm tới Python 3.12 và cài theo 2 giai đoạn: torch cu118 trước (T4 là GPU trước dòng RTX 50), rồi phần còn lại. Cell này mất ~5-10 phút.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd /content/rvc
!apt-get -qq install -y ffmpeg unzip libsndfile1 libportaudio2
!pip install -q torch==2.7.1+cu118 torchaudio==2.7.1+cu118 --index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://pypi.org/simple
!pip install -q -r requirments_cu118_py312.txt --index-url https://pypi.org/simple --extra-index-url https://download.pytorch.org/whl/cu118
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 2. Tải model nền (hubert + rmvpe + pretrained v2 + mute)

In [ ]:
%cd /content/rvc
BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"
for f in ["config.json", "preprocessor_config.json", "pytorch_model.bin"]:
    !wget -q -P assets/hubert_base {BASE}/hubert_base/{f}
!wget -q -P assets/rmvpe {BASE}/rmvpe.pt
for f in ["f0G40k.pth", "f0D40k.pth"]:
    !wget -q -P assets/pretrained_v2 {BASE}/pretrained_v2/{f}
!wget -q -O /tmp/mute.zip {BASE}/mute.zip && mkdir -p logs && unzip -q -o /tmp/mute.zip -d logs
!ls assets/hubert_base assets/rmvpe assets/pretrained_v2 && ls logs/mute

## 3. Upload dataset.zip (thư mục dataset đã cắt sẵn)

In [ ]:
from google.colab import files

uploaded = files.upload()  # chọn dataset.zip
!mkdir -p /content/dataset && unzip -q -o -j $(ls *.zip | head -1) -d /content/dataset
!ls /content/dataset | head
!ls /content/dataset | wc -l

## 4. Train

Mở link public gradio in ra bên dưới, vào tab **Train**:

| Trường | Giá trị |
|---|---|
| Experiment name | `phien-singer` |
| Target sample rate | `40k` |
| Model có pitch guidance | `true` (bắt buộc cho hát) |
| Version | `v2` |
| Trainset folder | `/content/dataset` |
| f0 method | `rmvpe_gpu` |
| Batch size | `8` (T4); dataset < 5 phút thì để `4` |
| Total epochs | `150-250`; dataset < 5 phút thì `100` |
| Save frequency | `25` |
| Cache dataset to GPU | `No` nếu dataset > 15 phút |

Bấm lần lượt: **Process data** → **Feature extraction** → **Train model** → **Train feature index**.

Mẹo: dừng khi loss đi ngang; train quá lâu sẽ overfit (giọng nghe méo, mất tự nhiên).

In [ ]:
%cd /content/rvc
!python webui.py --colab --pycmd python

## 5. Tải model về máy

Bỏ 2 file này vào thư mục `models/` của repo `rvc-cover-vi`.

In [ ]:
import glob

from google.colab import files

NAME = "phien-singer"
pth = f"/content/rvc/assets/weights/{NAME}.pth"
index = sorted(glob.glob(f"/content/rvc/logs/{NAME}/added_*.index")) + sorted(
    glob.glob(f"/content/rvc/assets/indices/*{NAME}*.index")
)
print(pth, index)
files.download(pth)
if index:
    files.download(index[0])